In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.text_normalize import normalize_text

In [2]:
def check(raw, expected):
    got = normalize_text(raw)
    assert got == expected, f"expected {expected!r}, got {got!r}"

check("\u0643\u0627\u0644\u0627", "\u06A9\u0627\u0644\u0627")      # Arabic kaf -> Persian kaf
check("خوب   بود\n\nعالی", "خوب بود عالی")                           # spaces and line breaks
check("می\u200cخواهم", "می\u200cخواهم")                              # half-space must stay
check("a\u200bb\u200fc", "abc")                                      # invisible characters removed
check("قیمت ۱۲۰۰ و ٣٤ و 56", "قیمت 1200 و 34 و 56")                # digits unified
check("سایت https://x.com/a?b=1 خوب", "سایت خوب")                   # URL removed
check("عالی 😍", "عالی 😍")                                          # emoji kept
check("نه، اصلاً خوب نیست", "نه، اصلاً خوب نیست")                  # negation kept
check("  \t خوبي\u00a0و\r\nكيفيت ", "خوبی و کیفیت")                 # mixed case
check(None, "")
print("all checks passed")

all checks passed


In [3]:
import duckdb

con = duckdb.connect()
con.sql("""
    CREATE OR REPLACE TEMP VIEW comments AS
    SELECT * FROM read_csv('../data/digikala-comments.csv', all_varchar=true)
""")

sample = con.sql(r"""
    SELECT id, body
    FROM comments
    WHERE body LIKE '%ي%' OR body LIKE '%ك%'
       OR regexp_matches(body, '[\x{06F0}-\x{06F9}]')
    LIMIT 8
""").df()

sample["normalized"] = sample["body"].apply(normalize_text)
sample

,id,body,normalized
0,45354820,به نظرم چیز خوب و خفنیه مخصوصا با قیمت ۱۰ ت \r...,به نظرم چیز خوب و خفنیه مخصوصا با قیمت 10 ت من...
1,17911442,من چند رنگش رو سفارش دادم.خیلی خوش رنگ بودن تو...,من چند رنگش رو سفارش دادم.خیلی خوش رنگ بودن تو...
2,47241861,اینا رو بیرون دونه ای ۵ یا هفت میفروشن \r\nالا...,اینا رو بیرون دونه ای 5 یا هفت میفروشن الان 6 ...
3,25986578,خیلی خوشگل و کاربردی هستش و اندازه مناسبی داره...,خیلی خوشگل و کاربردی هستش و اندازه مناسبی داره...
4,29582488,سلام من سایه آکوآ رنگ ۷۴ سفارش دادم طبق نظرلت ...,سلام من سایه آکوآ رنگ 74 سفارش دادم طبق نظرلت ...
5,19945372,خیلی نسبت به قیمتش فقط دوستان بدونین این ماسک ...,خیلی نسبت به قیمتش فقط دوستان بدونین این ماسک ...
6,53894544,خوبه\r\nهمشون سالم بودن\r\nتعداد درست ارسال شد...,خوبه همشون سالم بودن تعداد درست ارسال شده بود ...
7,52130458,تعداد توپک ها زیاده و هر کدوم حجم خوبی دارن\nق...,تعداد توپک ها زیاده و هر کدوم حجم خوبی دارن قی...


In [4]:
from duckdb.sqltypes import VARCHAR

con.create_function("normalize_text", normalize_text, [VARCHAR], VARCHAR)

con.sql("SELECT normalize_text('خوبي   و كيفيت ۱۲۳') AS result").show()

┌──────────────────┐
│      result      │
│     varchar      │
├──────────────────┤
│ خوبی و کیفیت 123 │
└──────────────────┘



In [5]:
from pathlib import Path

INTERIM = Path("..") / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)
NORMALIZED = INTERIM / "comments_normalized.parquet"

con.sql(f"""
    COPY (
        SELECT id, product_id, rate, recommendation_status, is_buyer,
               body, normalize_text(body) AS body_norm
        FROM comments
    ) TO '{NORMALIZED.as_posix()}' (FORMAT PARQUET)
""")

CountBIGINT
6156289


In [6]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(*) FILTER (WHERE body <> body_norm) AS changed_rows,
           COUNT(*) FILTER (WHERE body_norm LIKE '%ي%' OR body_norm LIKE '%ك%') AS arabic_letters_left,
           COUNT(*) FILTER (WHERE body_norm = '') AS empty_after
    FROM read_parquet('{NORMALIZED.as_posix()}')
""").show()

┌─────────┬──────────────┬─────────────────────┬─────────────┐
│ n_rows  │ changed_rows │ arabic_letters_left │ empty_after │
│  int64  │    int64     │        int64        │    int64    │
├─────────┼──────────────┼─────────────────────┼─────────────┤
│ 6156289 │      1624138 │                   0 │           0 │
└─────────┴──────────────┴─────────────────────┴─────────────┘



In [7]:
con.sql(f"""
    WITH b AS (
        SELECT body_norm, COUNT(*) AS n
        FROM read_parquet('{NORMALIZED.as_posix()}')
        GROUP BY body_norm
    )
    SELECT COUNT(*) AS distinct_bodies,
           COUNT(*) FILTER (WHERE n > 1) AS bodies_repeated,
           SUM(n) FILTER (WHERE n > 1)   AS rows_in_repeated_bodies
    FROM b
""").show()

┌─────────────────┬─────────────────┬─────────────────────────┐
│ distinct_bodies │ bodies_repeated │ rows_in_repeated_bodies │
│      int64      │      int64      │         int128          │
├─────────────────┼─────────────────┼─────────────────────────┤
│         4027268 │          193458 │                 2322479 │
└─────────────────┴─────────────────┴─────────────────────────┘



In [8]:
con.sql(f"""
    WITH t AS (
        SELECT body_norm,
               COUNT(DISTINCT recommendation_status) AS n_labels,
               COUNT(*) AS n
        FROM read_parquet('{NORMALIZED.as_posix()}')
        WHERE recommendation_status <> 'nan'
        GROUP BY body_norm
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS repeated_labelled_bodies,
           COUNT(*) FILTER (WHERE n_labels > 1) AS with_conflicting_labels
    FROM t
""").show()

┌──────────────────────────┬─────────────────────────┐
│ repeated_labelled_bodies │ with_conflicting_labels │
│          int64           │          int64          │
├──────────────────────────┼─────────────────────────┤
│                   169985 │                   31393 │
└──────────────────────────┴─────────────────────────┘



## Normalization rules (src/text_normalize.py)

Each rule follows a prevalence count from notebook 01:
- Arabic yeh/kaf -> Persian yeh/kaf (48,719 and 25,363 rows).
- Persian and Arabic-Indic digits -> Latin digits (provisional; to be re-checked against the model tokenizer).
- Invisible characters other than the half-space (U+200C) are removed (1,005 rows). The half-space is kept (193,672 rows).
- URLs are removed (9 rows).
- Whitespace runs, tabs and line breaks become one space (210,707 and 567,096 rows).
- Emojis, punctuation, negation words and stop words are kept unchanged.